In [ ]:
# get persistent cookies using jupyter notebook
# modified from get_chosen_v2.2_publish by teruteru
# welcome to follow teruteru! he is very cute!

import re
import json
import time
import pickle
import qrcode
import requests
import threading
import http.cookies
from websocket import create_connection
from IPython.display import display

# Disable cookie validation
http.cookies._is_legal_key = lambda _: True
session = requests.session()


def wss_monitor(wss, update_qr_code_callback, login_callback):
    while True:
        try:
            r = wss.recv()
        except Exception:
            break
        if not r:
            break
        j = json.loads(r)
        t = j["type"]
        if t == "UPDATE_QR_CODE":
            p = j["payload"]
            sig = p["sig"]
            ts = p["ts"]
            update_qr_code_callback(ts, sig)
        elif t == "LOGIN":
            login_callback()
            return


def get_wss(uuid, update_qr_code_callback, login_callback):
    wss = create_connection(f"wss://jaccount.sjtu.edu.cn/jaccount/sub/{uuid}")
    t = threading.Thread(
        target=wss_monitor, args=(wss, update_qr_code_callback, login_callback)
    )
    t.start()
    return wss, t


def send_update_qr_code(wss):
    wss.send('{ "type": "UPDATE_QR_CODE" }')


def update_qr_code(ts, sig):
    qr = qrcode.QRCode(box_size=4, border=4)
    qr.add_data(
        f"https://jaccount.sjtu.edu.cn/jaccount/confirmscancode?uuid={uuid}&ts={ts}&sig={sig}"
    )
    qr.make(fit=True)
    img = qr.make_image(fill_color="black", back_color="white")
    display(img)


def login():
    print("Having scanned successfully😎")


def match_uuid(body):
    match = re.search(r'href="jaccount://login\?uuid=([0-9a-zA-Z\-]+)"', body)
    if match:
        return match.group(1)
    else:
        print("Error: regular expression matching failed😮")
        exit()


def save(session: requests.Session, file_path: str = "cookies"):
    cookies = session.cookies.get_dict("jaccount.sjtu.edu.cn")
    if len(cookies) == 0:
        return
    with open(file_path, "wb") as f:
        pickle.dump(cookies, f)


res = session.get("https://i.sjtu.edu.cn/jaccountlogin")
body = str(res.content)
uuid = match_uuid(body)
wss, t = get_wss(uuid, update_qr_code, login)
time.sleep(0.2)

print("Please scan the QRCode to login📱")
send_update_qr_code(wss)
t.join()

res2 = session.get(f"https://jaccount.sjtu.edu.cn/jaccount/expresslogin?uuid={uuid}")
if res2.url.startswith("https://jaccount.sjtu.edu.cn/jaccount/jalogin"):
    print("Error: authorization failed, the page has been redirected to login🤒")
    exit()
if not res2.url.startswith("https://i.sjtu.edu.cn/"):
    print("Error: authorization failed🤒")
    exit()
save(session)
print("Login successfully!😋")